# 08. 분석대상지역 선정

시설 데이터의 **미제공 지역을 0으로 간주하지 않고 제외**하기 위해,
실제 데이터가 공통으로 존재하는 지역의 교집합을 만듭니다.

- 메인 분석: 신호등 + 횡단보도 + 어린이사고 데이터가 모두 있는 지역
- 추가 분석: 메인 분석 지역 중 과속방지턱 데이터까지 있는 지역

**출력**
- `../data_processed/main_analysis_49_regions.csv`
- `../data_processed/speed_bump_analysis_35_regions.csv`

In [ ]:
import pandas as pd

crosswalk = pd.read_csv(
    "../data_processed/crosswalk_count_final.csv"
)
traffic = pd.read_csv(
    "../data_processed/traffic_light_count_final.csv"
)
speed_bump = pd.read_csv(
    "../data_processed/speed_bump_count_final.csv"
)
accident = pd.read_csv(
    "../data_clean/accident_2020_2025_child_clean.csv"
)

print("횡단보도 지역:", len(crosswalk))
print("신호등 지역:", len(traffic))
print("과속방지턱 지역:", len(speed_bump))

## 1) 신호등 + 횡단보도 공통 지역

In [ ]:
main_regions = crosswalk.merge(
    traffic,
    on=["SIDO_NM", "SIGUNGU_NM"],
    how="inner"
)

print("시설 2종 공통 지역:", len(main_regions))

## 2) 어린이사고 데이터 존재 여부 확인

In [ ]:
def normalize_region_name(x):
    if pd.isna(x):
        return x

    x = str(x).strip()
    x = x.replace("강원도 ", "강원특별자치도 ")
    x = x.replace("전라북도 ", "전북특별자치도 ")

    if x == "세종특별자치시":
        x = "세종특별자치시 세종시"

    return x

accident_regions = (
    accident[["sido_sgg_nm"]]
    .drop_duplicates()
    .copy()
)

accident_regions["sido_sgg_nm"] = (
    accident_regions["sido_sgg_nm"]
    .apply(normalize_region_name)
)

accident_regions[["SIDO_NM", "SIGUNGU_NM"]] = (
    accident_regions["sido_sgg_nm"]
    .str.split(" ", n=1, expand=True)
)

accident_regions = accident_regions[
    ["SIDO_NM", "SIGUNGU_NM"]
].drop_duplicates()

main_49 = main_regions.merge(
    accident_regions,
    on=["SIDO_NM", "SIGUNGU_NM"],
    how="inner"
)

print("신호등 + 횡단보도 + 어린이사고 공통 지역:", len(main_49))

## 3) 과속방지턱 추가분석 지역

In [ ]:
sub_35 = main_49.merge(
    speed_bump,
    on=["SIDO_NM", "SIGUNGU_NM"],
    how="inner"
)

print("과속방지턱까지 있는 지역:", len(sub_35))
display(main_49.head())
display(sub_35.head())

In [ ]:
main_49.to_csv(
    "../data_processed/main_analysis_49_regions.csv",
    index=False,
    encoding="utf-8-sig"
)

sub_35.to_csv(
    "../data_processed/speed_bump_analysis_35_regions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")